In [1]:
import os
import re
import warnings
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d

try:
    from numpy.exceptions import RankWarning
except ImportError:
    from numpy import RankWarning

warnings.simplefilter('ignore', RankWarning)
warnings.filterwarnings('ignore')

In [2]:
DATASET_PATH    = "../data/dataset.csv"
FILLED_PATH     = "../data/filled_dataset.csv"
SUBMISSION_PATH = "../submissions/submission.csv"

df = pd.read_csv(DATASET_PATH)

feature_cols = [c for c in df.columns if c not in ['datetime', 'underlying_price']]
ce_cols = sorted([c for c in feature_cols if c.endswith('CE')])
pe_cols = sorted([c for c in feature_cols if c.endswith('PE')])

def parse_strike(col: str) -> int:
    m = re.search(r'\d{5}', col)
    return int(m.group()) if m else 0

ce_strikes = np.array([parse_strike(c) for c in ce_cols], dtype=float)
pe_strikes = np.array([parse_strike(c) for c in pe_cols], dtype=float)

print(f"Dataset: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"CE / PE contracts: {len(ce_cols)} / {len(pe_cols)}")

Dataset: 975 rows x 30 columns
CE / PE contracts: 14 / 14


In [3]:
# --- Tuned hyperparameters ---
N_NEIGHBORS  = 5      
CONV_THRESH  = 1e-8   
SPIKE_THRESH = 1.5    
W_INTERIOR   = 0.85   # Slightly higher trust in the weighted quadratic
W_EDGE       = 0.50   
IV_FLOOR     = 0.001
IV_CAP       = 5.0    

def _weighted_local_quad_estimate(obs_strikes, obs_ivs, target):
    """
    Fit a local convex quadratic, but heavily weight the strikes 
    closest to the target missing value to reduce spatial interpolation error.
    """
    distances = np.abs(obs_strikes - target)
    dist_order = np.argsort(distances)[:N_NEIGHBORS]
    
    if len(dist_order) < 3:
        return np.nan
        
    local_s = obs_strikes[dist_order]
    local_v = obs_ivs[dist_order]
    local_d = distances[dist_order]
    
    # Inverse distance weighting ( + epsilon to prevent division by zero)
    weights = 1.0 / (local_d + 1e-3)
    
    coeffs = np.polyfit(local_s, local_v, deg=2, w=weights)
    
    if coeffs[0] > CONV_THRESH:          
        return float(np.polyval(coeffs, target))
    return np.nan

def _damped_boundary_extrap(obs_strikes, obs_ivs, target):
    """
    Exponentially decay the extrapolation slope to prevent massive blowups
    on sparse wings .
    """
    decay_factor = 250.0  
    
    if target < obs_strikes[0]:
        slope = (obs_ivs[1] - obs_ivs[0]) / (obs_strikes[1] - obs_strikes[0])
        dist = obs_strikes[0] - target
        damped_dist = decay_factor * (1 - np.exp(-dist / decay_factor))
        return float(obs_ivs[0] - slope * damped_dist)
    else:
        slope = (obs_ivs[-1] - obs_ivs[-2]) / (obs_strikes[-1] - obs_strikes[-2])
        dist = target - obs_strikes[-1]
        damped_dist = decay_factor * (1 - np.exp(-dist / decay_factor))
        return float(obs_ivs[-1] + slope * damped_dist)

def impute_wing_weighted(ivs: np.ndarray, strikes: np.ndarray) -> np.ndarray:
    out = ivs.copy()
    obs_mask = np.isfinite(out)

    if obs_mask.sum() < 2:
        return out   

    obs_s = strikes[obs_mask]
    obs_v = out[obs_mask]
    miss_idx = np.where(~obs_mask)[0]

    if len(miss_idx) == 0:
        return out

    is_spike = float(np.nanmax(obs_v)) > SPIKE_THRESH
    lin_fn = interp1d(obs_s, obs_v, kind='linear', bounds_error=False, fill_value=np.nan)

    for j in miss_idx:
        target = strikes[j]
        v_lin  = float(lin_fn(target))
        is_interior = np.isfinite(v_lin)
        
        if not is_interior:
            v_lin = _damped_boundary_extrap(obs_s, obs_v, target)

        if not is_spike:
            # Deploy the weighted WLS quadratic
            q = _weighted_local_quad_estimate(obs_s, obs_v, target)
            if np.isfinite(q):
                w = W_INTERIOR if is_interior else W_EDGE
                final = (1.0 - w) * v_lin + w * q
            else:
                final = v_lin
        else:
            final = v_lin

        out[j] = float(np.clip(final, IV_FLOOR, IV_CAP))

    return out

In [4]:
df_filled = df.copy()
all_iv_cols = ce_cols + pe_cols

# 1. High-Precision Spatial Pass
for idx in df_filled.index:
    ce_vals = df_filled.loc[idx, ce_cols].values.astype(float)
    pe_vals = df_filled.loc[idx, pe_cols].values.astype(float)
    df_filled.loc[idx, ce_cols] = impute_wing_weighted(ce_vals, ce_strikes)
    df_filled.loc[idx, pe_cols] = impute_wing_weighted(pe_vals, pe_strikes)

# 2. Hard Temporal Safety Net (Only triggers if a wing is 100% missing)
df_filled = df_filled.sort_values('datetime').reset_index(drop=True)

rolling_past = df_filled[all_iv_cols].rolling(window=3, min_periods=1).mean()
df_filled[all_iv_cols] = df_filled[all_iv_cols].fillna(rolling_past)
df_filled[all_iv_cols] = df_filled[all_iv_cols].ffill(axis=0).fillna(0.0)

print(f"Final missing count: {df_filled[all_iv_cols].isna().sum().sum()}")
df_filled.to_csv(FILLED_PATH, index=False)
print(f"Filled dataset saved \u2192 {FILLED_PATH}")

Final missing count: 0
Filled dataset saved → ../data/filled_dataset.csv


In [5]:
SEPARATOR = "||"

def generate_solution(original_path: str, filled_path: str, output_path: str):
    original = pd.read_csv(original_path)
    filled   = pd.read_csv(filled_path)

    feat_cols = [c for c in original.columns if c not in ["datetime", "underlying_price"]]
    rows = []

    for col in feat_cols:
        was_missing = original[col].isna()
        for idx in original.index[was_missing]:
            dt  = original.loc[idx, "datetime"]
            uid = f"{dt}{SEPARATOR}{col}"
            val = filled.loc[idx, col]
            rows.append({"id": uid, "value": val})

    solution = pd.DataFrame(rows, columns=["id", "value"])
    solution = solution.sort_values("id").reset_index(drop=True)

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    solution.to_csv(output_path, index=False)
    print(f"Submission saved \u2192 {output_path}  ({len(solution)} rows)")
    return solution

sol = generate_solution(DATASET_PATH, FILLED_PATH, SUBMISSION_PATH)

Submission saved → ../submissions/submission.csv  (5460 rows)
